## All in-built function and classes IMPORTS

In [1]:
import numpy as np
import pandas as pd 
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader
from torch import nn
import torch.nn.functional as F

In [2]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

## All defined functions and classes IMPORTS

In [3]:
from data_processing.readDataset import dataGrabber
from data_processing.preProcessing import DataPreprocessor
from data_processing.data_preparation import pytorch_lightning_dataModule
from data_processing.data_preparation import pytorch_lightning_module
from Neural_Network.feed_forward import pytorch_model
from data_processing.LSTM_dataloader import LSTM_data
from data_processing.LSTM_dataloader import TimeSeriesDataset
from Neural_Network.LSTM import LSTM
from Neural_Network.LSTM import run
from Physics_Based.Acceleration_Model import acceleration
from Physics_Based.Bicycle_Model import Bicycle_Model

## Step 1: Read Dataset

In [4]:
dataset_path = 'dataset/data/'
recording_id_sel=[]
for i in range (32):
    recording_id_sel.append(f'{i}')
    
data_obj = dataGrabber(dataset_path)

data_obj.recording_id = recording_id_sel
data_obj.read_csv_with_recordingID()

track_data_raw = data_obj.get_tracks_data()
track_meta_data_raw = data_obj.get_tracksMeta_data()

## Step 2: PreProcessing

Step 2.1: Label Encoding

In [5]:
preprocessor = DataPreprocessor(track_meta_data_raw)
encode=preprocessor.label_encode()

for i in range(len(track_meta_data_raw)):
    element=track_meta_data_raw[i]
    element['encoded_class'] = element['class'].map(encode)
    track_meta_data_raw.append(element)

bicycle: 1
car: 3
pedestrian: 0
truck_bus: 2


In [6]:
new_track_data=[]
for i in range (len(track_data_raw)):
    element=track_data_raw[i]
    element2=track_meta_data_raw[i]
    tracklifetime=element['trackLifetime'].tolist()
    classes=[]
    k=0
    for j in range (len(tracklifetime)):
        if tracklifetime[j]==0:
            if k < len(element2.iloc[:, -1]):
                class_value=element2.iloc[k,-1]
                k=k+1
        classes.append(class_value)
    element['encoded class']=classes
    new_track_data.append(element)

Step 2.2: Downsampling & Normalization

In [7]:

output_of_downsample=[]
output_of_normalise=[]
for i in range (len(new_track_data)):
    element=new_track_data[i]
    element = element.drop(['lonVelocity','latVelocity','lonAcceleration','latAcceleration'],axis=1)
    preprocessor = DataPreprocessor(element)
    downsample = preprocessor.downsample(0.5)  # Downsample the data to 50% of its original size
    output_of_downsample.append(downsample)
    normalise=preprocessor.normalize() 
    output_of_normalise.append(normalise)

Step 2.3 Getting Required data

In [8]:
xvelocity_list=[]
yvelocity_list=[]
xacceleration_list=[]
yacceleration_list=[]
xcenter_list=[]
ycenter_list=[]
heading_list=[]
encoded_class_list=[]

for i in range (len(output_of_downsample)):
    element=output_of_downsample[i]
    xvelocity=element['xVelocity'].tolist()
    xvelocity_list.extend(xvelocity)
    yvelocity=element['yVelocity'].tolist()
    yvelocity_list.extend(yvelocity)
    xcenter=element['xCenter'].tolist()
    xcenter_list.extend(xcenter)
    ycenter=element['yCenter'].tolist()
    ycenter_list.extend(ycenter)
    xacceleration=element['xAcceleration'].tolist()
    xacceleration_list.extend(xacceleration)
    yacceleration=element['yAcceleration'].tolist()
    yacceleration_list.extend(yacceleration)
    heading=element['yAcceleration'].tolist()
    heading_list.extend(heading)
    encoded_class=element['encoded class'].tolist()
    encoded_class_list.extend(encoded_class)

X=[xvelocity_list, yvelocity_list,encoded_class_list,heading_list,xacceleration_list,yacceleration_list]
y=[xcenter_list,ycenter_list]

x=np.array(X)
x=x.T
y=np.array(y)
y=y.T

## Step 3: Physics Based Model

## Acceleration Model

In [9]:
pred = acceleration.forward(x,y)
pred_final = acceleration.predictionmatrix(pred,y)
loss= acceleration.loss_function(y,pred_final)

In [10]:
print(loss)

tensor(0.5201, dtype=torch.float64)


## Bicycle Model

In [11]:
T_s=1/25
L=2
xpred, ypred, heading_pred=Bicycle_Model.bicycle_model(heading_list, y,x[:,0],x[:,1],T_s,L)

print("Loss in x_direction: ", np.mean(xpred-xcenter_list))
print("Loss in y_direction:",np.mean(ypred-ycenter_list))
print("Loss in heading", np.mean(heading_pred-heading_list))

Loss in x_direction:  0.0070589242093409225
Loss in y_direction: -0.0007938504515250179
Loss in heading -6.368764913767155


## Step 4: Neural Network

## Feed Forward Neural Network

In [12]:
input_dim=5
hidden_dim=20
output_dim=y.shape[1]
batch_size=20
x=x[:,:3]

xarray=np.hstack((x,y))
xarray=np.hstack((xarray,y))

In [13]:
# Create a PyTorch Lightning data module and set it up
my_data_module = pytorch_lightning_dataModule(xarray, batch_size)  # Create a data module object
my_data_module.setup(part_training=0.8)  # Set up the data module by splitting the dataset

# Create a PyTorch model and wrap it in a PyTorch Lightning module
my_pytorch_model = pytorch_model(input_dim, hidden_dim, output_dim)  # Create a PyTorch model object
my_pytorch_lightning_module = pytorch_lightning_module(my_pytorch_model, input_dim, output_dim)  # Wrap the model in a Lightning module

In [14]:
trainer = pl.Trainer(max_epochs=1,
                     fast_dev_run=False,
                     devices="auto",
                     accelerator="auto",
                    #  logger = logger,
                     check_val_every_n_epoch=1,
                     #precision='16-mixed',
                     )
torch.set_float32_matmul_precision('medium')
trainer.fit(my_pytorch_lightning_module, my_data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name          | Type          | Params
------------------------------------------------
0 | pytorch_model | pytorch_model | 1.4 K 
------------------------------------------------
1.4 K     Trainable params
0         Non-trainable params
1.4 K     Total params
0.006     Total estimated model params size (MB)


Epoch 0: 100%|██████████| 276976/276976 [20:37<00:00, 223.79it/s, v_num=9] 

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 276976/276976 [20:37<00:00, 223.78it/s, v_num=9]


In [15]:
train_mse = trainer.logged_metrics['train_loss']
val_mse = trainer.logged_metrics['val_loss']

print("Train MSE:", train_mse)
print("Validation MSE:", val_mse)

Train MSE: tensor(0.0055)
Validation MSE: tensor(0.0126)


## LSTM

In [39]:
input_dim=5
hidden_dim=20
output_dim=y.shape[1]
batch_size=20
datasize=0.8
num_layers=3

data_loader=LSTM_data(x,y,datasize)
X_train, Y_train, X_test, y_test=data_loader.converter()

train_dataset = TimeSeriesDataset(X_train, Y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for _, batch in enumerate(train_loader):
    x_batch, y_batch = batch[0].to(device), batch[1].to(device)
    print(x_batch.shape, y_batch.shape)
    break

model=LSTM(input_dim,hidden_dim,num_layers)
model

torch.Size([20, 20, 5]) torch.Size([20, 20, 2])


LSTM(
  (lstm): LSTM(5, 20, num_layers=3, batch_first=True)
  (fc): Linear(in_features=20, out_features=2, bias=True)
)

In [40]:
learning_rate = 0.0001
num_epochs = 1
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [41]:
def train_one_epoch():
    model.train(True)
    print(f'Epoch: {epoch + 1}')
    running_loss = 0.0

    for batch_index, batch in enumerate(train_loader):
        if batch_index == len(train_loader) - 1:
            break
        x_batch, y_batch = batch[0].to(device), batch[1].to(device)

        output = model(x_batch)
        # print("output",output.shape)
        loss = F.mse_loss(output, y_batch)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch_index % 100 == 99:  # print every 100 batches
            avg_loss_across_batches = running_loss / 100
            print('Batch {0}, Loss: {1:.3f}'.format(batch_index+1,
                                                    avg_loss_across_batches))
            running_loss = 0.0
    print("Train Loss", loss)
    # log("train_loss", loss)
    print()

In [42]:
def validate_one_epoch():
    model.train(False)
    running_loss = 0.0

    for batch_index, batch in enumerate(test_loader):
        if batch_index == len(test_loader) - 1:
            break
        x_batch, y_batch = batch[0].to(device), batch[1].to(device)
        
        with torch.no_grad():
  
            output = model(x_batch)
            # print("output",output.shape)
            loss = F.mse_loss(output, y_batch)
            running_loss += loss.item()

    avg_loss_across_batches = running_loss / len(test_loader)

    print('Val Loss: {0:.3f}'.format(avg_loss_across_batches))
    print('***************************************************')
    print("Val Loss", loss)
    print()

In [43]:
for epoch in range(num_epochs):
    train_one_epoch()
    validate_one_epoch()

Epoch: 1


C:\Users\Tanmay\AppData\Local\Temp\ipykernel_21656\1168418636.py:13: UserWarning: Using a target size (torch.Size([20, 20, 2])) that is different to the input size (torch.Size([20, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(output, y_batch)


Batch 100, Loss: 0.033


Batch 200, Loss: 0.015
Batch 300, Loss: 0.012
Batch 400, Loss: 0.012
Batch 500, Loss: 0.013
Batch 600, Loss: 0.012
Batch 700, Loss: 0.012
Batch 800, Loss: 0.012
Batch 900, Loss: 0.012
Batch 1000, Loss: 0.012
Batch 1100, Loss: 0.012
Batch 1200, Loss: 0.012
Batch 1300, Loss: 0.012
Batch 1400, Loss: 0.012
Batch 1500, Loss: 0.012
Batch 1600, Loss: 0.012
Batch 1700, Loss: 0.012
Batch 1800, Loss: 0.012
Batch 1900, Loss: 0.012
Batch 2000, Loss: 0.012
Batch 2100, Loss: 0.012
Batch 2200, Loss: 0.012
Batch 2300, Loss: 0.012
Batch 2400, Loss: 0.012
Batch 2500, Loss: 0.012
Batch 2600, Loss: 0.012
Batch 2700, Loss: 0.012
Batch 2800, Loss: 0.012
Batch 2900, Loss: 0.012
Batch 3000, Loss: 0.012
Batch 3100, Loss: 0.012
Batch 3200, Loss: 0.012
Batch 3300, Loss: 0.012
Batch 3400, Loss: 0.012
Batch 3500, Loss: 0.012
Batch 3600, Loss: 0.012
Batch 3700, Loss: 0.012
Batch 3800, Loss: 0.012
Batch 3900, Loss: 0.012
Batch 4000, Loss: 0.012
Batch 4100, Loss: 0.012
Batch 4200, Loss: 0.012
Batch 4300, Loss: 0.012


C:\Users\Tanmay\AppData\Local\Temp\ipykernel_21656\3293488582.py:14: UserWarning: Using a target size (torch.Size([20, 20, 2])) that is different to the input size (torch.Size([20, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(output, y_batch)


Val Loss: 0.016
***************************************************
Val Loss tensor(0.0153)

